In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from src.datasets import get_data_loaders, get_labels
from src.train import train
from src.evaluate import evaluate
from src.models import make_model
from src.utils import set_seed, load_model
from src.metrics import evaluate_mc_metrics

set_seed(42)

In [ ]:
dataset_name = 'CIFAR10'
model_name = 'resnet18'
device = torch.device('mps' if torch.cuda.is_available() else 'cpu')
batch_size = 128
epochs = 50
learning_rate = 0.001
patience = 5
mc_passes = 20
num_classes = 10

In [ ]:
train_loader, val_loader, test_loader = get_data_loaders(
    name=dataset_name, batch_size=batch_size
)
model = make_model(model_name)
model = model.to(device)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Train
checkpoint_path = f"{model_name}_{dataset_name}_best.pth"
train(model, train_loader, val_loader, optimizer, criterion,
      epochs=epochs, device=device, patience=patience, checkpoint_path=checkpoint_path)

model = load_model(model, checkpoint_path, device=device)

# Evaluate with MC Dropout
_, _, all_probs = evaluate(model, test_loader, criterion, device=device, mc_dropout=True, mc_passes=mc_passes)

# Get ground truth labels
labels = get_labels(name=dataset_name, test=True)

# Convert logits to numpy
all_probs = all_probs.cpu().numpy()

# Evaluate metrics
evaluate_mc_metrics(all_probs, labels)
